<a href="https://colab.research.google.com/github/sakusdev/convenient-colabnotebooks/blob/main/MuScripter_Large_%E6%8E%A8%E8%AB%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MuScriptor Large - Google Colab
# Audio / Video -> MIDI
# ============================================================

# 依存関係
!pip -q install -U muscriptor huggingface_hub

import subprocess
import torch

from pathlib import Path
from getpass import getpass

from google.colab import files
from huggingface_hub import login, whoami
from muscriptor import TranscriptionModel


# ============================================================
# 1. GPU確認
# ============================================================

print("=== GPU ===")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPUが見つかりません。\n"
        "Colabの「ランタイム」→「ランタイムのタイプを変更」→ GPU を選択してください。"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("GPU :", gpu_name)
print(f"VRAM: {vram_gb:.1f} GB")


# ============================================================
# 2. Hugging Faceログイン
# ============================================================

print("\n=== Hugging Face Login ===")

hf_token = None

# Colab Secrets の HF_TOKEN を優先
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

# Secretsになければ手入力
if not hf_token:
    hf_token = getpass("HF_TOKEN: ")

login(
    token=hf_token,
    add_to_git_credential=False
)

try:
    account = whoami(token=hf_token)
    print("Logged in as:", account["name"])
except Exception:
    print("Hugging Face login complete.")


# ============================================================
# 3. MuScriptor Large 読み込み
# ============================================================

print("\n=== Loading MuScriptor Large ===")

model = TranscriptionModel.load_model(
    weights_path="large",
    device="cuda",
    dtype="float16",
)

print("MuScriptor Large loaded!")


# ============================================================
# 4. 音声 / 動画ファイルをアップロード
# ============================================================

print("\n=== Upload audio / video ===")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("ファイルがアップロードされませんでした。")

input_file = next(iter(uploaded.keys()))
input_path = Path(input_file)

print("Input:", input_path)


# ============================================================
# 5. WAVへ変換
# ============================================================

wav_path = Path("/content/muscriptor_input.wav")

print("\n=== Converting to WAV ===")

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", str(input_path),

        # 動画の場合は映像を無視
        "-vn",

        # WAV
        "-acodec", "pcm_s16le",

        str(wav_path),
    ],
    check=True,
)

print("WAV:", wav_path)


# ============================================================
# 6. MuScriptor Large 推論
# ============================================================

output_midi = Path("/content/muscriptor_large.mid")

print("\n=== Transcribing with MuScriptor Large ===")
print("Quality mode: prelude_forcing=True")
print("batch_size=1")
print("処理には時間がかかる場合があります。")


with torch.inference_mode():

    midi_bytes = model.transcribe_to_midi(
        wav_path,

        # prelude forcing使用時は1必須
        batch_size=1,

        # まずは高速寄り
        beam_size=1,

        # EOSが出なくても処理を継続
        no_eos_is_ok=True,

        # チャンク境界の品質改善
        prelude_forcing=True,
    )


# ============================================================
# 7. MIDI保存
# ============================================================

output_midi.write_bytes(midi_bytes)

print("\n=== DONE ===")
print("Output:", output_midi)

midi_size_kb = output_midi.stat().st_size / 1024

print(f"MIDI size: {midi_size_kb:.1f} KiB")


# ============================================================
# 8. MIDIをダウンロード
# ============================================================

print("\n=== Download MIDI ===")

files.download(str(output_midi))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
=== GPU ===
GPU : Tesla T4
VRAM: 14.6 GB

=== Hugging

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.47GB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

MuScriptor Large loaded!

=== Upload audio / video ===


...

[muscriptor] mel-spec (1 × 80000 samples): 0.695s
[muscriptor] encode conditions (total): 0.780s
[muscriptor] mel-spec (1 × 80000 samples): 0.000s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.000s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.000s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.001s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.001s
[muscriptor] encode conditions (total): 0.002s
[muscriptor] mel-spec (1 × 80000 samples): 0.001s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.000s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.000s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 80000 samples): 0.001s
[muscriptor] encode conditions (total): 0.001s
[muscriptor] mel-spec (1 × 800

[muscriptor] generate total: 515.39s
[muscriptor] transcribe total: 515.87s (276.7s audio)



=== DONE ===
Output: /content/muscriptor_large.mid
MIDI size: 59.2 KiB

=== Download MIDI ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>